# Langfuse API Levels

This notebook shows three Langfuse instrumentation styles using the same small calculator example:

- **Decorator API**: easiest default; `@observe()` creates traces and nested spans automatically.
- **Context manager API**: explicit `with` blocks define the trace hierarchy.
- **Low-level API**: manual observation lifecycle for maximum control.

Before running the cells, make sure your environment has `OPENAI_API_KEY`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and `LANGFUSE_BASE_URL` configured.

## 1. Decorator API

Use `@observe()` when you want Langfuse to create traces and spans from normal Python function calls. Nested observed functions become nested observations in the Langfuse UI.

In [1]:
# Decorator-level API: use normal Python functions and annotate them with
# @observe() so Langfuse creates traces/spans automatically.
from langfuse import observe, Langfuse
from openai import OpenAI
from dotenv import load_dotenv

# Load OpenAI and Langfuse credentials from .env, if present.
load_dotenv()

# Use the regular OpenAI client for the LLM call.
client = OpenAI()

# Langfuse client is used here to enrich the current span and flush data.
langfuse = Langfuse()


@observe()  # Creates a trace/span automatically.
def calculator(expression: str) -> str:
    """Single calculation - becomes a span when called from another @observe function."""
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a very accurate calculator. You output only the result of the calculation.",
            },
            {"role": "user", "content": expression},
        ],
    )

    # Add metadata and tags to the active span so it can be filtered in Langfuse.
    langfuse.update_current_span(
        metadata={"project": "decorator_example", "tags": ["calculator", "math"]},
    )

    return completion.choices[0].message.content


@observe()  # Nested observed function = parent span with child spans.
def process_calculations(expressions: list[str]) -> list[str]:
    """Process multiple calculations - each calculator() call becomes a child span."""
    results = []
    for expr in expressions:
        # Each calculator() call is captured as a child span under this trace.
        result = calculator(expr)
        results.append(f"{expr} = {result}")
    return results


# Running this creates one trace for the batch and child spans for each calculation.
expressions = ["123 + 456", "789 * 2", "100 / 4"]
results = process_calculations(expressions)

print("Results:")
for result in results:
    print(f"  {result}")

# Flush is important in notebooks so buffered observations are sent immediately.
langfuse.flush()

Results:
  123 + 456 = 579
  789 * 2 = 1578
  100 / 4 = 25


## 2. Context Manager API

Use context managers when you want explicit control over the active span or generation, but still want Langfuse to manage nesting and cleanup when each `with` block exits.

In [2]:
# Context-manager API: explicitly open trace/span/generation scopes with
# `with` blocks, while Langfuse manages the active current object.
from langfuse import Langfuse
from openai import OpenAI
from dotenv import load_dotenv

# Load OpenAI and Langfuse credentials from .env, if present.
load_dotenv()

# Regular OpenAI client performs the model call.
client = OpenAI()

# Langfuse client creates and updates the trace hierarchy.
langfuse = Langfuse()

expression = "123 + 456 * 2"

# Create the root span. Because it is the outermost active span, it becomes
# the root of a new trace in Langfuse.
with langfuse.start_as_current_observation(
    name="calculator_context_manager",
    as_type="span",
) as trace:

    # Add a child span for preprocessing/validation work.
    with langfuse.start_as_current_observation(
        name="input_validation",
        as_type="span",
    ) as validation_span:
        # Update the currently active span with input and output data.
        langfuse.update_current_span(
            input={"expression": expression},
            output={"status": "valid"},
        )

    # Add a generation for the actual LLM call. Generations are the right
    # Langfuse object for model requests because they track model, tokens, and cost.
    with langfuse.start_as_current_observation(
        name="llm_calculation",
        as_type="generation",
        model="gpt-4o-mini",
        input=[
            {
                "role": "system",
                "content": "You are a very accurate calculator. You output only the result of the calculation.",
            },
            {"role": "user", "content": expression},
        ],
    ) as generation:
        # The OpenAI call itself still uses the regular OpenAI SDK.
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "You are a very accurate calculator. You output only the result of the calculation.",
                },
                {"role": "user", "content": expression},
            ],
        )
        result = completion.choices[0].message.content

        # Attach the LLM output, token usage, and metadata to the active generation.
        langfuse.update_current_generation(
            output=result,
            usage_details={
                "input": completion.usage.prompt_tokens,
                "output": completion.usage.completion_tokens,
            },
            metadata={"project": "context_manager_example"},
        )

    # Attach final request-level output and tags to the root span.
    trace.update(
        output={"result": result},
        tags=["calculator", "context_manager"],
    )

print(f"{expression} = {result}")

# Flush is important in notebooks so buffered observations are sent immediately.
langfuse.flush()

123 + 456 * 2 = 123 + 912 = 1035


## 3. Low-Level API

Use the low-level API when you need maximum control over observation lifecycle, custom timing, scores, events, or manually linked child observations.

In [3]:
# Low-level API: manually create, update, end, and link Langfuse objects.
# This gives maximum control, but requires more lifecycle code.
from langfuse import Langfuse
from openai import OpenAI
from dotenv import load_dotenv
import time

# Load OpenAI and Langfuse credentials from .env, if present.
load_dotenv()

# Regular OpenAI client performs the model call.
client = OpenAI()

# Langfuse client is used directly to create spans, generations, scores, and events.
langfuse = Langfuse()

expression = "123 + 456 * 2"

# Create the root span. Starting a root span creates a new trace automatically.
root_span = langfuse.start_observation(
    name="calculator_low_level",
    as_type="span",
    input={"expression": expression},
    metadata={"project": "low_level_example"},
)

# Keep the trace ID so later objects, such as scores, can be attached to this trace.
trace_id = root_span.trace_id

# Create and manually finish a child span for preprocessing/validation.
preprocessing_span = root_span.start_observation(
    name="preprocessing",
    as_type="span",
    input={"raw_expression": expression},
)

# Update the child span with the validation result.
preprocessing_span.update(
    output={"validated_expression": expression, "status": "valid"},
)

# Low-level spans must be ended explicitly.
preprocessing_span.end()

# Track custom timing around the model call.
start_time = time.time()

# The LLM request itself is still a normal OpenAI SDK call.
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a very accurate calculator. You output only the result of the calculation.",
        },
        {"role": "user", "content": expression},
    ],
)

end_time = time.time()
result = completion.choices[0].message.content

# Create the generation manually as another child observation so we can control
# its name, model, parameters, input, output, usage, status, and lifecycle.
generation = root_span.start_observation(
    name="llm_calculation",
    as_type="generation",
    model="gpt-4o-mini",
    model_parameters={"temperature": 1.0},
    input=[
        {
            "role": "system",
            "content": "You are a very accurate calculator. You output only the result of the calculation.",
        },
        {"role": "user", "content": expression},
    ],
)

# Update the generation with output, token usage, severity level, and status.
generation.update(
    output=result,
    usage_details={
        "input": completion.usage.prompt_tokens,
        "output": completion.usage.completion_tokens,
        "total": completion.usage.total_tokens,
    },
    level="DEFAULT",  # Options: "DEBUG", "DEFAULT", "WARNING", "ERROR"
    status_message="Calculation completed successfully",
)

# Low-level generations must also be ended explicitly.
generation.end()

# Update and end the root span with final request-level output and custom timing.
root_span.update(
    output={"result": result},
    metadata={"duration_ms": (end_time - start_time) * 1000},
)
root_span.end()

# Attach an evaluation score to the trace.
langfuse.create_score(
    trace_id=trace_id,
    name="accuracy",
    value=1.0,
    comment="Correct calculation verified",
)

# Create a point-in-time event for debugging or audit-style logging.
event = langfuse.create_event(
    name="calculation_complete",
    trace_context={"trace_id": trace_id},
    input={"expression": expression},
    output={"result": result},
    metadata={"duration_ms": (end_time - start_time) * 1000},
)

print(f"{expression} = {result}")
print(f"Duration: {(end_time - start_time) * 1000:.2f}ms")
print(f"Tokens used: {completion.usage.total_tokens}")
print(f"Trace ID: {trace_id}")

# Flush is important in notebooks so buffered observations are sent immediately.
langfuse.flush()

123 + 456 * 2 = 123 + 912 = 1035
Duration: 1414.78ms
Tokens used: 42
Trace ID: f325ee4971d49ee94c3f00df4f9571c1
